# v01 · Prototype: bijective base-N + flag-byte packet
`PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/infinite-data.py` (git first commit 2026-04-18; position in the sequence inferred from the code, see wiki index).

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


## The file cannot be imported as-is
It reads `PTOL_ROOT` (undefined) at module level. The classes above that line are extracted verbatim and executed; nothing else is changed.

In [2]:
src = open(os.path.join(PROJ, "PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/infinite-data.py")).read()
cut = src.index("# TODO:SETTINGS")
ns = {}
try:
    exec(compile(src, "infinite-data.py", "exec"), {})
except NameError as e:
    print("whole file:", type(e).__name__, e)
exec(compile(src[:cut], "infinite-data.py[:cut]", "exec"), ns)
HW = ns["HyperWebster"]()
print("classes extracted:", [k for k in ns if k[0].isupper()])

whole file: NameError name 'PTOL_ROOT' is not defined
classes extracted: ['HyperWebster', 'UniversalCoordinateSystem']


## The bijection: address 0 is the first character; addresses are shortlex ranks

In [3]:
chars = HW.char_list
print("N =", HW.N)
for s in ["`", "1", "`1", "``"]:
    print(repr(s), "->", HW.point_to_text(s))
# shortlex: every one-character string precedes every two-character string
assert HW.point_to_text(chars[-1]) + 1 == HW.point_to_text(chars[0]*2)
for s in ["Hello, World!", "python3", CORPUS[:500]]:
    assert HW.regenerate_text(HW.point_to_text(s)) == s
print("round trips exact")

N = 97
'`' -> 0
'1' -> 1
'`1' -> 98
'``' -> 97
round trips exact


## The 256-bit packet: 8-bit flag + 248-bit coordinate
Capacity of the 248-bit coordinate field, computed exactly: the longest length L such that *every* string of length ≤ L has an address below 2²⁴⁸.

In [4]:
N = HW.N
def strings_up_to(L): return (N**(L+1) - N)//(N-1)
L = 0
while strings_up_to(L+1) <= 2**248: L += 1
print("every string up to length", L, "fits in 248 bits;", "length", L+1, "does not (all of them)")
print("log2(97) =", round(math.log2(N), 4), "bits/char ->", round(248/math.log2(N), 2), "chars by division")
assert HW.point_to_text("~"*L) < 2**248

every string up to length 37 fits in 248 bits; length 38 does not (all of them)
log2(97) = 6.5999 bits/char -> 37.58 chars by division


## `decode_packet` calls a method the class does not have

In [5]:
U = ns["UniversalCoordinateSystem"](HW)
packet = (0 << 248) | HW.point_to_text("hello")
try:
    U.decode_packet(packet)
except AttributeError as e:
    print("AttributeError:", e)
print("flag 1 ->", U.decode_packet((1 << 248) | 5))
print("flag 2 ->", U.decode_packet(2 << 248))

AttributeError: 'HyperWebster' object has no attribute 'locate_text'
flag 1 -> Bitmap Mode: Indexing 5 on the XZ Plane.
flag 2 -> Irrational Mode: Constant Expansion triggered.
